# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Louai-Mohamed/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

**Lane:** Lane 2 — *Refresh / Content Opportunity Scoring* (`docs/ml-intern-dataset-and-lane-guide.md`).

**One-line frame:** *for a content editor with about 50 review slots a week, deciding which page to refresh
first, we build a ranked action queue from trailing-90-day search and engagement signals, scoring each page
by how likely its visibility is to be declining. A wrong pick costs editor hours; the output is decision
support, never a promise that a refresh will recover a page.*

Sections 1–5 answer the five framing questions, each with the code that backs it. The self-check at the end
records what this framing deliberately does **not** claim.

## 1. My lane as an ML task (type)

**Ranking / scoring — not classification, and not clustering.**

The decision is not "is this page declining: yes or no?". It is **"which 50 of 30,000 pages does a human
open first?"** — an ordering problem. So the deliverable is a ranked queue: every page gets a score, the
queue is sorted by it, and the reviewer works down from the top until their capacity runs out.

A classifier does the arithmetic underneath (the label is binary), but its probability is used **as a
ranking score**, never as a yes/no cutoff. The threshold that matters is the reviewer's capacity, not 0.5 —
a page at 0.51 and a page at 0.99 are different decisions only if one makes the top 50.

Why not the other three: not **clustering** (I am not looking for archetypes — the decision is already
known); not plain **classification** (nobody acts on a 0.5 threshold, they act on the top of a list); and not
Lane 1 signal analysis, which stops at "what correlates" — this lane has to end in an ordered action list.

In [1]:
import os
import sys

if "google.colab" in sys.modules:            # Colab: bring the repo in so the paths resolve
    if not os.path.isdir("flyrank-ml-internship"):
        !git clone --depth 1 https://github.com/Louai-Mohamed/flyrank-ml-internship.git
    os.chdir("flyrank-ml-internship")
else:                                        # local: walk up to the repo root
    while os.path.isdir("..") and not os.path.isdir("data/raw"):
        os.chdir("..")

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

REVIEW_CAPACITY = 50          # pages one reviewer can actually work through in a week
print(f"inventory:  {len(df):,} pages")
print(f"capacity:   {REVIEW_CAPACITY} pages/week -> {len(df) / REVIEW_CAPACITY:,.0f} weeks to touch every page once")
print(f"consequence: the output is an ORDER over the top {REVIEW_CAPACITY / len(df):.2%} of the inventory,")
print("             not a yes/no label for each page.")

inventory:  30,000 pages
capacity:   50 pages/week -> 600 weeks to touch every page once
consequence: the output is an ORDER over the top 0.17% of the inventory,
             not a yes/no label for each page.


## 2. Target or proxy

**Capstone target — the one I actually want:** a *forward* decline. Impressions in the 30 days **after** the
decision point fall more than 20% below the 30 days before it, among pages with enough volume to matter
(≥ 100 impressions in the pre-window). Measured on `fact_content_daily_performance`, with the feature window
ending exactly where the target window starts.

**Starter proxy — the one I can build today:** `is_declining_label = (trend_direction == "down")`, i.e.
trailing `impressions_last_30d` versus `impressions_prev_30d` worse than −20%.

**Labelling the label honestly.** The starter target is *defined by a rule over the trailing window*, not an
observed future outcome. That is acceptable for learning the pipeline and for a first ranking, but it has two
consequences I have to write down rather than hide:

1. **It lives inside the feature window.** The primary metric is a trailing 90-day aggregate, and the label is
   computed from the last 60 days *of that same aggregate*. The next cells prove the label can be rebuilt from
   ordinary-looking window columns at 99.98% agreement — which is exactly why `trend_pct` and
   `trend_direction` are banned as features.
2. **It does not separate decline from its look-alikes** — consolidation, seasonality, SERP/AI click loss, or
   plain noise (lane guide §7). The warehouse version adds a minimum-volume floor, a persistence check and
   per-client windows, so a client's seasonal dip is not labelled as content decay.

The capstone therefore swaps the proxy for the forward target; everything else about the framing stays.
Nothing here claims a refresh *causes* recovery — that needs an experiment this data cannot provide.

In [2]:
# What the starter label is actually made of
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print("label source — share of pages per bucket:")
print((df["trend_direction"].value_counts(normalize=True) * 100).round(1).to_string())
print("\nlabel counts:", df["is_declining_label"].value_counts().to_dict())
print("base rate:", round(df["is_declining_label"].mean(), 3))

df[["content_id", "impressions_prev_30d", "impressions_last_30d", "trend_pct",
    "trend_direction", "is_declining_label"]].head(5)

label source — share of pages per bucket:
trend_direction
down      54.2
stable    19.9
up        14.6
new        7.5
flat       3.8

label counts: {1: 16262, 0: 13738}
base rate: 0.542


,content_id,impressions_prev_30d,impressions_last_30d,trend_pct,trend_direction,is_declining_label
0,content_304f48230142,987,578,-41.4,down,1
1,content_a1fb4e703a9e,5915,2501,-57.7,down,1
2,content_9aa793d4d895,6089,2382,-60.9,down,1
3,content_331d6c4de07b,4206,3626,-13.8,stable,0
4,content_d99b7a2d90ca,6452,4211,-34.7,down,1


In [3]:
import numpy as np

# Proof that the label is reproducible from columns that look like ordinary features
prev, last, tp = df["impressions_prev_30d"], df["impressions_last_30d"], df["trend_pct"]
rebuilt = np.select(
    [(prev == 0) & (last > 0), (prev == 0) & (last == 0), tp > 20, tp < -20],
    ["new", "flat", "up", "down"], default="stable",
)
agree = (rebuilt == df["trend_direction"]).mean()
print(f"rebuilt trend_direction from the raw window columns: {agree:.4%} agreement")
print(f"disagreeing rows: {int((rebuilt != df['trend_direction']).sum())} of {len(df):,} (rounding at the ±20% edge)")
print("-> the label is inside the observable columns. Never feed trend_* to a model.\n")

timeline = pd.DataFrame([
    {"window": "days -60 to -30", "starter columns": "impressions_prev_30d, clicks_prev_30d",
     "usable at prediction time?": "yes, before the label window"},
    {"window": "days -30 to 0", "starter columns": "impressions_last_30d, trend_pct",
     "usable at prediction time?": "NO, this IS the starter label"},
    {"window": "days 0 to +30", "starter columns": "(not in the starter slice)",
     "usable at prediction time?": "the honest label lives here"},
])
for row in timeline.itertuples(index=False):
    print(f"  {row[0]:<16}{row[1]:<46}{row[2]}")

rebuilt trend_direction from the raw window columns: 99.9800% agreement
disagreeing rows: 6 of 30,000 (rounding at the ±20% edge)
-> the label is inside the observable columns. Never feed trend_* to a model.

  days -60 to -30 impressions_prev_30d, clicks_prev_30d         yes, before the label window
  days -30 to 0   impressions_last_30d, trend_pct               NO, this IS the starter label
  days 0 to +30   (not in the starter slice)                    the honest label lives here


## 3. Success metric

**Precision@50, with the base rate printed beside it.**

Why 50: that is the capacity the decision assumes. A reviewer works down the queue until the week's slots are
gone, so `Precision@50` = *of the 50 pages at the top, what share are really declining* — it prices exactly
the thing that wastes editor time, a wrong page inside the top 50.

Numbers I report **next to** it, never instead of it:

- **Base rate (54.2% here).** An uninformative ranking already scores 0.542 at Precision@50. Every score is
  read against that floor — and note the reference pipeline's hand baseline lands at **0.24**, i.e. *below*
  the base rate at the very top: its weighted-percentile blend over-ranks big visible pages that are not
  actually declining. That is the honest size of the first baseline, and the reason there is work to do.
- **ROC-AUC / average precision** — quality over the whole ranking, which is the only fair view when the top
  is one large block of tied scores.
- **Coverage** — the share of pages a rule actually ranks at all. A rule that fires on 17 of 30,000 pages
  cannot be compared at Precision@50; the comparison is tie-ordering dressed up as skill.
- **Tie-aware averaging** — with only four distinct tree scores, Precision@50 is decided by tie breaks, so I
  average it over random tie resolutions instead of trusting one `argsort`.

**What "good" means, decided before training:** at least **2x the hand baseline's Precision@50 on clients
the model has never seen**, with the gain larger than the split-to-split spread. The starter proxy below
does **not** clear that bar — a depth-2 tree gets ≈0.59 against ≈0.48, a +0.10 gain inside a ±0.11 spread —
so it is reported as **directional**, and the capstone has to clear the full bar on the forward target.

In [4]:
base = df["is_declining_label"].mean()
k = REVIEW_CAPACITY
print(f"base rate: {base:.3f} -> a random top-{k} already scores Precision@{k} = {base:.3f}")
print(f"capacity:  {k} of {len(df):,} pages = {k / len(df):.2%} of the inventory\n")

print(f"{'ranking':<28}{'Precision@50':>13}{'right of 50':>13}")
for name, value in [("random (base rate)", base),
                    ("reference hand baseline", 0.24),
                    ("reference random forest", 0.68)]:
    print(f"{name:<28}{value:>13.3f}{round(value * k):>13}")

print("\nthe same numbers as the reviewer experiences them:")
print(f"  baseline queue -> {round(0.24 * k)} of {k} slots spent on a page that is actually declining")
print(f"  forest queue   -> {round(0.68 * k)} of {k}")
print(f"  coin flip      -> {round(base * k)} of {k}")

base rate: 0.542 -> a random top-50 already scores Precision@50 = 0.542
capacity:  50 of 30,000 pages = 0.17% of the inventory

ranking                      Precision@50  right of 50
random (base rate)                  0.542           27
reference hand baseline             0.240           12
reference random forest             0.680           34

the same numbers as the reviewer experiences them:
  baseline queue -> 12 of 50 slots spent on a page that is actually declining
  forest queue   -> 34 of 50
  coin flip      -> 27 of 50


## 4. The unit of analysis, as a real dataframe

**One row = one content item (a page) for one client, aggregated over one trailing 90-day window.** Every
number on that row describes that page: impressions, clicks, CTR, average position, freshness, and the trend
label.

Two consequences that shape everything downstream:

- the row is a **window snapshot**, not a day, so "declining" is always relative to that window;
- rows are **not independent** — pages from the same client share a site, an audience and a season, which is
  why every evaluation in this repo holds out *clients*, never random rows.

On the warehouse the grain becomes `report_date × client × content` (~79M rows, one per page per day) and a
lane query aggregates it back to this same page × window shape — so this dataframe is the right mental model
at every scale.

In [5]:
print("shape:", df.shape)
print("one row = one content item (page) for one client, over one trailing 90-day window\n")
print("content_id unique:", df["content_id"].is_unique)
print("clients:", df["client_id"].nunique())
per_client = df.groupby("client_id").size()
print("pages per client: min", per_client.min(), "| median", int(per_client.median()),
      "| max", per_client.max())
print("content types:", df["content_type"].value_counts().to_dict())

columns_that_carry_the_lane = ["content_id", "client_id", "content_type", "impressions_90d", "ctr",
                               "avg_position", "days_since_last_update", "content_age_days",
                               "trend_direction", "is_declining_label"]
df.sort_values("impressions_90d", ascending=False)[columns_that_carry_the_lane].head(5)

shape: (30000, 45)
one row = one content item (page) for one client, over one trailing 90-day window

content_id unique: True
clients: 32
pages per client: min 3 | median 567 | max 7008
content types: {'keyword article': 27207, 'feedly article': 2096, 'comparison article': 697}


,content_id,client_id,content_type,impressions_90d,ctr,avg_position,days_since_last_update,content_age_days,trend_direction,is_declining_label
6653,content_5fe46e04994d,client_4e07408562,keyword article,517715,0.14,4.2,104,537,down,1
17812,content_aaef01a50def,client_19581e27de,keyword article,517109,0.25,5.4,22,445,stable,0
26844,content_8c19996aa890,client_4e07408562,keyword article,509252,0.15,2.5,20,445,down,1
19636,content_2cb567c3c89b,client_6208ef0f77,keyword article,497727,0.10,22.2,48,153,up,0
21819,content_4c36c775b818,client_4e07408562,keyword article,463103,0.41,2.3,20,445,down,1


In [6]:
def provisional_action(row):
    """Sketch only: what an ACTION column looks like on each row. Hand rules, no model yet."""
    if row["impressions_90d"] >= 500 and row["days_since_last_update"] >= 180:
        return "refresh"
    if row["impressions_90d"] >= 250 and 0 < row["word_count"] < 1200:
        return "expand"
    if row["impressions_90d"] >= 500 and 0 < row["avg_position"] <= 20 and row["ctr"] < 0.5:
        return "fix_ctr"
    return "monitor"


df["sketch_action"] = df.apply(provisional_action, axis=1)
print("what each queued row will carry (the score comes later, from the model):")
print(df["sketch_action"].value_counts().to_string())
print(f"\nshare of the inventory that would be actioned at all: "
      f"{(df['sketch_action'] != 'monitor').mean():.1%}")

df.loc[df["sketch_action"] != "monitor",
       ["content_id", "content_type", "impressions_90d", "avg_position", "ctr",
        "days_since_last_update", "is_declining_label", "sketch_action"]].head(5)

what each queued row will carry (the score comes later, from the model):
sketch_action
monitor    20170
fix_ctr     9731
expand        82
refresh       17

share of the inventory that would be actioned at all: 32.8%


,content_id,content_type,impressions_90d,avg_position,ctr,days_since_last_update,is_declining_label,sketch_action
3,content_331d6c4de07b,keyword article,11751,6.2,0.49,22,0,fix_ctr
5,content_d4084a4bc775,keyword article,3970,8.5,0.03,20,1,fix_ctr
9,content_c27558df2b0c,keyword article,1240,4.9,0.16,104,1,fix_ctr
16,content_78bd1d4a1d4d,keyword article,13848,8.9,0.15,104,1,fix_ctr
17,content_761a44afda12,keyword article,9449,7.3,0.07,22,1,fix_ctr


## 5. Why ML beats a fixed rule here

Three measured reasons — the cell below recomputes all of them live, so nothing here is a quotation:

1. **The obvious rule barely fires.** `days_since_last_update ≥ 180 AND impressions ≥ 500` flags 17 of 30,000
   pages (0.06% coverage). A rule that skips 99.94% of the inventory is not a ranking — it is a filter with an
   empty queue, and it cannot be beaten *or* trusted at Precision@50.
2. **The signal is interaction-shaped, not threshold-shaped.** The Week-1 discovery in this repo found a CTR
   gap between content types that exists only for pages under ~180 days old and mostly vanishes after that;
   the reference pipeline's strongest features are `days_with_impressions`, log impressions, `avg_position` and
   `content_age_days` — four scales, four shapes, interacting. A hand rule needs one cutoff per variable per
   context, and every one of those cutoffs is a policy choice that shifts by tier (lane guide §11).
3. **Measured lift on clients the model never saw.** Across five client-holdout splits a depth-2 tree
   reaches ROC-AUC **0.647 ± 0.061** against **0.500 ± 0.001** for the one-line rule, and Precision@50
   **0.585 ± 0.108** against **0.484 ± 0.094** (base rate 0.468 ± 0.101). The AUC gain is more than twice
   its own spread, so the ranking carries real signal; the top-50 gain (+0.10) is about one spread wide,
   so at the very top of the queue this is **directional, not a proven win**. The reference forest's 0.68
   in-sample against its 0.24 baseline points the same way — the capstone repeats the comparison on a
   forward target before claiming more.

**Where a rule wins, honestly:** at the very top of the list a sharp human rule can be excellent, and if the
pattern were one clean threshold the right answer would be an if-statement, not a model. ML earns its place
here because there are many weak interacting signals and an ordering to produce — not because a model is the
goal. It also has to stay readable: reason codes on every row, so the reviewer can disagree with the score.

In [7]:
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier

y = df["is_declining_label"].values
rule_score = (((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500))
              .astype(int).values)


def p50_tie_aware(scores, labels, k=50, draws=100, seed=0):
    """Precision@k with ties broken at random, averaged — needed because 4 distinct scores = 4 blocks."""
    rng = np.random.default_rng(seed)
    s = np.asarray(scores, dtype=float)
    return float(np.mean([
        labels[np.argsort(-(s + rng.random(len(s)) * 1e-9))[:k]].mean() for _ in range(draws)
    ]))


features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

N_SPLITS = 5
rows = []
for split, (train_idx, test_idx) in enumerate(
        GroupShuffleSplit(n_splits=N_SPLITS, test_size=0.2, random_state=42)
        .split(df, groups=df["client_id"]), start=1):
    tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
    tree.fit(X.iloc[train_idx], y[train_idx])
    tree_score = tree.predict_proba(X.iloc[test_idx])[:, 1]
    rows.append({
        "split": split,
        "held-out clients": df["client_id"].iloc[test_idx].nunique(),
        "held-out pages": len(test_idx),
        "rule P@50": p50_tie_aware(rule_score[test_idx], y[test_idx], seed=split),
        "tree P@50": p50_tie_aware(tree_score, y[test_idx], seed=split),
        "rule AUC": roc_auc_score(y[test_idx], rule_score[test_idx]),
        "tree AUC": roc_auc_score(y[test_idx], tree_score),
        "base rate": y[test_idx].mean(),
    })

runs = pd.DataFrame(rows).set_index("split").round(3)
print(f"1-line rule coverage: {rule_score.mean():.3%} of pages "
      f"({int(rule_score.sum())} of {len(df):,})")
print(f"{N_SPLITS} client-holdout splits — 20% of the 32 clients held out each time, "
      f"~{runs['held-out pages'].mean():,.0f} pages per split\n")
print(f"averaged over the {N_SPLITS} splits (mean ± sd):")
print(f"  {'1-line rule':<14} P@50 {runs['rule P@50'].mean():.3f} ± {runs['rule P@50'].std():.3f}"
      f"   ROC-AUC {runs['rule AUC'].mean():.3f} ± {runs['rule AUC'].std():.3f}")
print(f"  {'depth-2 tree':<14} P@50 {runs['tree P@50'].mean():.3f} ± {runs['tree P@50'].std():.3f}"
      f"   ROC-AUC {runs['tree AUC'].mean():.3f} ± {runs['tree AUC'].std():.3f}")
print(f"  {'base rate':<14} P@50 {runs['base rate'].mean():.3f} ± {runs['base rate'].std():.3f}"
      "   (what a coin-flip ranking scores)")

runs

1-line rule coverage: 0.057% of pages (17 of 30,000)
5 client-holdout splits — 20% of the 32 clients held out each time, ~4,522 pages per split

averaged over the 5 splits (mean ± sd):
  1-line rule    P@50 0.484 ± 0.094   ROC-AUC 0.500 ± 0.001
  depth-2 tree   P@50 0.585 ± 0.108   ROC-AUC 0.647 ± 0.061
  base rate      P@50 0.468 ± 0.101   (what a coin-flip ranking scores)


,held-out clients,held-out pages,rule P@50,tree P@50,rule AUC,tree AUC,base rate
split,,,,,,,
1,7,6163,0.507,0.582,0.500,0.585,0.511
2,7,6365,0.604,0.756,0.500,0.717,0.595
3,7,4541,0.531,0.599,0.500,0.652,0.499
4,7,3432,0.376,0.488,0.501,0.695,0.342
5,7,2111,0.401,0.499,0.501,0.585,0.392


## Self-check

- [x] Every section above is filled — markdown thinking **and** the code that backs it
- [x] The notebook runs top to bottom with no errors (`Runtime → Run all`)
- [x] No client names, URLs, or private queries anywhere — only pseudonymous `content_id` / `client_id`
- [x] Claims use careful words: *observed, measured, directional, decision-support*
- [x] Committed under `work/notebooks/` and pushed

### What this framing does NOT claim

1. **No causality.** Refreshing a page is not shown to recover it; nothing here is a treatment effect. The
   queue allocates reviewer attention, full stop.
2. **No future-outcome label yet.** The starter proxy is computed from the same trailing window as the
   features (proved above: 99.98% reproducible). The capstone fixes this with a forward window on the
   warehouse; until then every number is an in-slice measurement.
3. **No "algorithm" claim.** These are observed associations in a 30,000-page anonymized slice of 32 clients —
   descriptive of these pages, not of Google.
4. **No precision promise.** With ~7 held-out clients per split, the top-50 comparison moves by ±0.1; the
   reported lift is a direction with a range, not a fixed number.
5. **The thresholds are mine, not universal.** 50 review slots, ≥100 impressions, −20% decline, ≥180 days
   stale — each is a policy choice that changes who lands in the queue, and each is stated where it is used.